# Random Forest
Εκπαιδεύουμε πολλά variations του μοντέλου με διαφορετικές υπερπαραμέτρους και βρίσκουμε το μοντέλο με τις καλύτερες υπερπαραμέτρους. Ύστερα, το εφαρμόζουμε πάνω στο test gold για να κάνουμε προβλέψεις και τις αποθηκεύουμε για να γίνουν evaluate μαζί με όλες τις άλλες 

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

print("1. Εκκίνηση SparkSession...")
spark = SparkSession.builder \
    .appName("RF_SMOTE_Corrected_GridSearch") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print("2. Φόρτωση και Feature Augmentation...")
train_gold = spark.read.parquet("../../data/train_gold_with_cluster.parquet")
test_gold = spark.read.parquet("../../data/test_gold_with_cluster.parquet")

train_gold = train_gold.withColumn("stroke", train_gold["stroke"].cast(DoubleType()))
test_gold = test_gold.withColumn("stroke", test_gold["stroke"].cast(DoubleType()))
train_gold = train_gold.withColumn("cluster", F.col("cluster").cast(DoubleType()))
test_gold = test_gold.withColumn("cluster", F.col("cluster").cast(DoubleType()))

assembler = VectorAssembler(inputCols=["features", "cluster"], outputCol="augmented_features")
train_augmented = assembler.transform(train_gold).drop("features").withColumnRenamed("augmented_features", "features")
test_augmented = assembler.transform(test_gold).drop("features").withColumnRenamed("augmented_features", "features")

# Κρατάμε ολόκληρο το SMOTE dataset (χωρίς undersampling)
train_augmented.cache()

print("3. Ορισμός Πλέγματος Παραμέτρων (Grid Search)...")
rf_base = RandomForestClassifier(featuresCol="features", labelCol="stroke", seed=22390225)

# ΟΙ ΑΛΛΑΓΕΣ:
# - maxDepth [3, 5, 7]: Πιο ρηχά δέντρα για αποφυγή αποστήθισης του SMOTE.
# - numTrees [50, 100, 150]: Τα 200 δέντρα συχνά ομαλοποιούν υπερβολικά τις προβλέψεις υπέρ της πλειοψηφικής κλάσης.
# - featureSubsetStrategy: Αναγκάζει τα δέντρα να επιλέγουν τυχαία υποσύνολα χαρακτηριστικών, αυξάνοντας το Recall της Class 1.
paramGrid = (ParamGridBuilder()
             .addGrid(rf_base.maxDepth, [3, 5, 7]) 
             .addGrid(rf_base.numTrees, [50, 100, 150])
             .addGrid(rf_base.featureSubsetStrategy, ["sqrt", "log2", "0.2"])
             .build())

# ΔΙΟΡΘΩΣΗ: Αλλαγή σε areaUnderROC για να μεγιστοποιήσουμε το Recall και τη συνολική διαχωριστική ικανότητα
evaluator = BinaryClassificationEvaluator(
    labelCol="stroke", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderROC"
)

cv = CrossValidator(estimator=rf_base,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    numFolds=3,
                    seed=22390225)

print("4. Εκτέλεση Cross-Validation στο ΠΛΗΡΕΣ SMOTE Dataset...")
cv_model = cv.fit(train_augmented)
best_rf = cv_model.bestModel

print("\n===========================================================")
print("[ΔΙΟΡΘΩΜΕΝΗ ΕΥΡΕΣΗ SMOTE] Οι βέλτιστες παράμετροι βρέθηκαν:")
print(f" -> maxDepth: {best_rf._java_obj.getMaxDepth()}")
print(f" -> numTrees: {best_rf._java_obj.getNumTrees()}")
print("===========================================================")

print("5. Παραγωγή και αποθήκευση προβλέψεων στο Test Set...")
rf_preds = best_rf.transform(test_augmented)

# Αποθήκευση ως το κεντρικό rf_predictions
rf_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../../data/rf_predictions.parquet")

spark.stop()
print("Ολοκληρώθηκε! Έχεις πλέον ένα RF που αντιστέκεται στο overfitting του SMOTE.")

1. Εκκίνηση SparkSession...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/11 18:05:17 WARN Utils: Your hostname, cachyos-x8664, resolves to a loopback address: 127.0.1.1; using 192.168.1.5 instead (on interface enp4s0)
26/06/11 18:05:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 18:05:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2. Φόρτωση και Feature Augmentation...
3. Ορισμός Πλέγματος Παραμέτρων (Grid Search)...
4. Εκτέλεση Cross-Validation στο ΠΛΗΡΕΣ SMOTE Dataset...


26/06/11 18:05:41 WARN DAGScheduler: Broadcasting large task binary with size 1367.9 KiB
26/06/11 18:05:42 WARN DAGScheduler: Broadcasting large task binary with size 1113.8 KiB
26/06/11 18:05:42 WARN DAGScheduler: Broadcasting large task binary with size 1367.9 KiB
26/06/11 18:05:43 WARN DAGScheduler: Broadcasting large task binary with size 1113.8 KiB
26/06/11 18:05:44 WARN DAGScheduler: Broadcasting large task binary with size 1367.9 KiB
26/06/11 18:05:44 WARN DAGScheduler: Broadcasting large task binary with size 1113.8 KiB
26/06/11 18:05:45 WARN DAGScheduler: Broadcasting large task binary with size 1264.7 KiB
26/06/11 18:05:45 WARN DAGScheduler: Broadcasting large task binary with size 2046.0 KiB
26/06/11 18:05:45 WARN DAGScheduler: Broadcasting large task binary with size 1649.1 KiB
26/06/11 18:05:46 WARN DAGScheduler: Broadcasting large task binary with size 1264.7 KiB
26/06/11 18:05:46 WARN DAGScheduler: Broadcasting large task binary with size 2046.0 KiB
26/06/11 18:05:47 WAR


[ΔΙΟΡΘΩΜΕΝΗ ΕΥΡΕΣΗ SMOTE] Οι βέλτιστες παράμετροι βρέθηκαν:
 -> maxDepth: 7
 -> numTrees: 100
5. Παραγωγή και αποθήκευση προβλέψεων στο Test Set...


26/06/11 18:06:35 WARN DAGScheduler: Broadcasting large task binary with size 1373.0 KiB


Ολοκληρώθηκε! Έχεις πλέον ένα RF που αντιστέκεται στο overfitting του SMOTE.
